In [1]:
import math
import os
import pickle
import numpy as np
import pandas as pd


working_dir = os.getenv('SCRATCHDIR_USER')
if working_dir is None:
    working_dir = os.getenv("HOME")
checkpoint_path = f"{working_dir}/checkpoints"

In [2]:
def to_percent(x: np.ndarray, precision: int = 1) -> np.ndarray:
    """Round the number for percent reporting"""
    scale = 10**(precision+2)
    return np.ceil(scale*x)/(scale/100)


def get_stats(
        val_contrib: np.ndarray, bad_points: np.ndarray
        ) -> tuple[np.ndarray, np.ndarray]:
    """Get mean and standard error of individual contributions"""
    means = []
    stds = []
    for i in range(len(val_contrib)):
        inv_n = (~bad_points[i]).sum() - 1
        if inv_n >= 0:
            means.append(val_contrib[i, ~bad_points[i]].mean())
        else:
            means.append(np.nan)
        if inv_n > 0:
            inv_n = (inv_n)**(-0.5)
            stds.append(val_contrib[i, ~bad_points[i]].std()*inv_n)
        else:
            stds.append(np.nan)
    return np.array(means), np.array(stds)


def get_contrib(
        val_loss: np.ndarray, s_val_loss: np.ndarray, split_name: str
        ) -> tuple[np.ndarray, np.ndarray]:
    """Get individual layers' contributions"""
    c_ind, s_ind = 0, -1
    if split_name[0] == "c":
        c_ind, s_ind = -1, 0
    val_err = val_loss[1:-1] - val_loss[s_ind]
    s_val_err = s_val_loss[1:-1] - s_val_loss[c_ind]
    bad_points = (val_err > 0) & (s_val_err > 0)

    val_contrib = (val_loss[:1] - val_loss[1:-1]) / (val_loss[:1] - val_loss[-1:])
    return get_stats(val_contrib, bad_points)


def get_sum_contrib(
        val_loss: np.ndarray, s_val_loss: np.ndarray, split_name: str
        ) -> tuple[np.ndarray, np.ndarray]:
    """Get individual layers' contributions"""
    c_ind, s_ind = 0, -1
    if split_name[0] == "c":
        c_ind, s_ind = -1, 0
    val_err = val_loss[1:-1] - val_loss[s_ind]
    s_val_err = s_val_loss[1:-1] - s_val_loss[c_ind]
    bad_points = (val_err > 0) & (s_val_err > 0)

    val_contrib = (val_loss[:1] - val_loss[1:-1]) / (val_loss[:1] - val_loss[-1:])
    sum_contrib = val_contrib.sum(0)[None, :]
    bad_points = bad_points.any(0)[None, :]
    return get_stats(sum_contrib, bad_points)


NAMES = {"v11": "VGG-11", "r18": "ResNet-18", "dtt": "DeiT-Ti"}
DATA_NAMES = {
    "c10m-w": "CIFAR-10 (weak)", "c10m-s": "CIFAR-10 (strong)",
    "wb": "Waterbirds", "ca": "CelebA"}
inv_n = (5-1)**(-0.5)

In [3]:
datasets = ["c10m-s", "wb", "ca"]
models = ["dtt-f", "r18-f", "v11-f"]
optimizers = ["adamw", "sgd"]

In [ ]:
contr = []

for split_name in ["ch", "sh"]:
    data_contr = []
    with open(f'{checkpoint_path}/{split_name}-err.pickle', 'rb') as f:
        val_losses = pickle.load(f)

    for data_name in datasets:
        for model_name in models:
            for optimizer_name in optimizers:
                val_contrib = val_losses[f"{data_name}-c-{model_name}-{optimizer_name}"]
                val_contrib = (val_contrib[:-1] - val_contrib[1:]) / (val_contrib[:1] - val_contrib[-1:])
                data_contr.append(val_contrib.mean(1))
    contr.append(np.concatenate(data_contr))

optim_labels = []
model_labels = []
data_labels = []
for data_name in datasets:
    for model_name in models:
        for optimizer_name in optimizers:
            optim_labels = optim_labels + [optimizer_name] * 6
        model_labels = model_labels + [model_name] * (6 * len(optimizers))
    data_labels = data_labels + [data_name] * (6 * len(optimizers) * len(models))
first_labels = ([1.0] + [0.0]*5) * (len(optimizers) * len(models) * len(datasets))
last_labels = ([0.0]*5 + [1.0]) * (len(optimizers) * len(models) * len(datasets))
layer_labels = [f"{i}:{i+1}" for i in range(6)] * (len(optimizers) * len(models) * len(datasets))

contr = pd.DataFrame({
    "fgt": contr[1], "enc": contr[0],
    "first": first_labels, "last": last_labels,
    "optimizer": optim_labels, "model": model_labels, "data": data_labels, "layer": layer_labels})
contr

In [ ]:
with open(f'{checkpoint_path}/sh-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)

base_err = []
optim_labels = []
data_labels = []
model_labels = []
for data_name in datasets:
    for model_name in models:
        for optimizer_name in optimizers:
            val_contrib = val_losses[f"{data_name}-c-{model_name}-{optimizer_name}"]
            base_err.append(val_contrib.mean(1)[[0, -1]])
            optim_labels = optim_labels + [optimizer_name]
        model_labels = model_labels + [model_name] * len(optimizers)
    data_labels = data_labels + [data_name] * (len(optimizers) * len(models))
base_err = np.stack(base_err)
base_err = pd.DataFrame(base_err)
base_err.columns = ["base_err", "base_err_s"]
base_err["data"] = data_labels
base_err["model"] = model_labels
base_err["optimizer"] = optim_labels
base_err

In [ ]:
import torch

def regress(x, y):
    inv_xx = torch.linalg.inv(torch.mm(x, x.T))
    w = torch.mm(torch.mm(y, x.T), inv_xx)
    res_y = y.sub(torch.mm(w, x)).square_()
    rsq = 1 - res_y.mean() / y.var()
    sigma = torch.einsum("li,ji,ki->ljk", res_y, x, x) * x.shape[1] / (x.shape[1] - x.shape[0])
    w_err = (inv_xx @ sigma @ inv_xx)
    return rsq, w, w_err


with open(f'{checkpoint_path}/exp-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)

r_squares = []
f_stats = []
weights = []
errors = []
num_obs = []

for exp_name in ("lr-up", "lr-down", "wd-up", "wd-down"):
    exp_res = []
    optim_labels = []
    model_labels = []
    data_labels = []
    layer_labels = []
    for data_name in datasets:
        for model_name in models:
            for optimizer_name in optimizers:
                val_contrib = val_losses[f"{data_name}-c-{model_name}-flex-{optimizer_name}-{exp_name}"]
                exp_res.append(val_contrib.mean(1)[:-1])
                optim_labels = optim_labels + [optimizer_name] * 6
            model_labels = model_labels + [model_name] * (6 * len(optimizers))
        data_labels = data_labels + [data_name] * (6 * len(optimizers) * len(models))
    layer_labels\
        = [f"{i}:{i+1}" for i in range(6)] * (len(optimizers)*len(models)*len(datasets))

    exp_res = pd.DataFrame({
        "err_s": np.concatenate(exp_res),
        "optimizer": optim_labels, "data": data_labels, "model": model_labels, "layer": layer_labels})

    reg_res = exp_res.merge(contr, on=["optimizer", "model", "data", "layer"])
    reg_res = reg_res.merge(base_err, on=["optimizer", "model", "data"])
    reg_res["inter"] = reg_res["fgt"] * reg_res["enc"]
    reg_res["enc2"] = reg_res["enc"]**2
    reg_res["fgt2"] = reg_res["fgt"]**2
    reg_res["eff_size"]\
        = (reg_res["base_err_s"]-reg_res["err_s"])\
        / (reg_res["base_err_s"]-reg_res["base_err"])

    # reg_res = reg_res[reg_res["data"] == "c10m-s"]

    reg_res["const"] = 1
    x = reg_res[["enc", "fgt", "inter", "enc2", "fgt2", "const", "last"]]
    x = torch.as_tensor(x.to_numpy().T)
    y = reg_res[["eff_size"]]
    y = torch.as_tensor(y.to_numpy().T)
    rsq, w, w_err = regress(x, y)
    k = 5
    r_squares.append(rsq.item())
    f_stats.append((w[0][:k] @ torch.linalg.inv(w_err[0][:k, :k]) @ w[0][:k]).item()/k)
    weights.append(w[0].tolist())
    errors.append(w_err[0].diag().sqrt().tolist())
    num_obs.append(len(reg_res))
    print(reg_res[reg_res["layer"] == "3:4"])


with open(f'{checkpoint_path}/fr-err.pickle', 'rb') as f:
    val_losses = pickle.load(f)

exp_res = []
optim_labels = []
model_labels = []
data_labels = []
layer_labels = []
for data_name in datasets:
    for model_name in models:
        for optimizer_name in optimizers:
            val_contrib = val_losses[f"{data_name}-c-{model_name}-{optimizer_name}-freeze"]
            exp_res.append(val_contrib.mean(1)[:-1])
            optim_labels = optim_labels + [optimizer_name] * 6
        model_labels = model_labels + [model_name] * (6 * len(optimizers))
    data_labels = data_labels + [data_name] * (6 * len(optimizers) * len(models))
layer_labels\
    = [f"{i}:{i+1}" for i in range(6)] * (len(optimizers)*len(models)*len(datasets))

exp_res = pd.DataFrame({
    "err_s": np.concatenate(exp_res),
    "optimizer": optim_labels, "data": data_labels, "model": model_labels, "layer": layer_labels})

reg_res = exp_res.merge(contr, on=["optimizer", "model", "data", "layer"])
reg_res = reg_res.merge(base_err, on=["optimizer", "model", "data"])
reg_res["inter"] = reg_res["fgt"] * reg_res["enc"]
reg_res["enc2"] = reg_res["enc"]**2
reg_res["fgt2"] = reg_res["fgt"]**2
reg_res["eff_size"]\
    = (reg_res["base_err_s"]-reg_res["err_s"])\
    / (reg_res["base_err_s"]-reg_res["base_err"])

# reg_res = reg_res[reg_res["data"] == "c10m-s"]

reg_res["const"] = 1
x = reg_res[["enc", "fgt", "inter", "enc2", "fgt2", "const", "last"]]
x = torch.as_tensor(x.to_numpy().T)
y = reg_res[["eff_size"]]
y = torch.as_tensor(y.to_numpy().T)
rsq, w, w_err = regress(x, y)
k = 5
r_squares.append(rsq.item())
f_stats.append((w[0][:k] @ torch.linalg.inv(w_err[0][:k, :k]) @ w[0][:k]).item()/k)
weights.append(w[0].tolist())
errors.append(w_err[0].diag().sqrt().tolist())
num_obs.append(len(reg_res))


EXP_NAMES = ["LR Up", "LR Down", "WD Up", "WD Down", "Freeze"]
REG_NAMES = ["Enc", "Fgt", r"Enc $\times$ Fgt", "Enc2", "Fgt2", "Const", "Last"]
print("&", " & ".join(EXP_NAMES), r"\\")
for i, name in enumerate(REG_NAMES):
    print(
        f"{REG_NAMES[i]} &",
        " & ".join(map(
            lambda x: f"${x:.2f}$",
            (weights[j][i] for j in range(len(weights))))),
        r"\\")
    print(
        "&",
        " & ".join(map(
            lambda x: f"$({np.ceil(x*100)/100:.2f})$",
            (errors[j][i] for j in range(len(errors))))),
        r"\\")

print("F-Stat &", " & ".join(map(lambda x: f"${x:.2f}$", f_stats)), r"\\")
print("R${}^2$ &", " & ".join(map(lambda x: f"${x:.2f}$", r_squares)), r"\\")
print("N &", " & ".join(map(lambda x: f"${x}$", num_obs)), r"\\")

In [ ]:
import scipy.stats as sps

sps.f.ppf(1-0.05, 5, 108-7)